# 🧠 NetlogRAG — Fine-tuning v2 (ispravljeni dataset)
**Diplomski rad** | Sveučilište Jurja Dobrile u Puli

## Što je bilo krivo u v1

Prva verzija dala je **100% accuracy na sva tri modela** — to nije bio dobar rezultat
nego znak da je zadatak bio trivijalan:

| Problem | Posljedica |
|---|---|
| Samo Tuesday fajl → 3 labele → 2 razine rizika | Binarna klasifikacija |
| Odredišni port bio prvi podatak u ulazu | FTP=21, SSH=22 → model naučio samo port |
| Predviđala se samo razina rizika | Premalo informacije |

Praktično: `if port in (21, 22): HIGH` dao bi istih 100%.

## Što je popravljeno u v2

Više CICIDS2017 dana pa se pojavljuju DoS, PortScan i web napadi, a s njima
i **MEDIUM** klasa. Port je **izbačen iz ulaza** — ostaju samo numeričke značajke
toka, pa model mora učiti ponašanje umjesto pamtiti broj porta. I model sad
predviđa **tip napada**, ne samo razinu rizika, što je izravno usporedivo
s XGBoost baselineom u projektu.

> ⚠️ Runtime → Change runtime type → **T4 GPU**


## 0. Provjera GPU-a

In [ ]:
import torch, subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else "❌ GPU nije dostupan")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")


## 1. Instalacija

In [ ]:
%%capture
!pip install unsloth datasets huggingface_hub
print("✅ Gotovo")


## 2. Konfiguracija

Mijenjaj `MODEL_ID` i `MODEL_NAME` za svaki od tri modela.
Za Phi-3.5 (3.8B) smanji `BATCH_SIZE` na 2 i podigni `GRAD_ACCUM` na 8.


In [ ]:
# ── MODEL ─────────────────────────────────────────────────────
MODEL_ID   = "meta-llama/Llama-3.2-1B-Instruct"
MODEL_NAME = "llama32-1b-netlograg-v2"

# Ostale opcije:
#   "HuggingFaceTB/SmolLM2-1.7B-Instruct"  → smollm2-1.7b-netlograg-v2
#   "microsoft/Phi-3.5-mini-instruct"      → phi35-mini-netlograg-v2   (BATCH_SIZE=2, GRAD_ACCUM=8)

# ── HUGGINGFACE ───────────────────────────────────────────────
HF_USERNAME = "lovro77"
HF_TOKEN    = ""      # ← zalijepi token, obriši nakon pokretanja
PUSH_TO_HUB = True

# ── HIPERPARAMETRI ────────────────────────────────────────────
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
MAX_SEQ_LENGTH = 512
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
LEARNING_RATE  = 2e-4
NUM_EPOCHS     = 3
WARMUP_RATIO   = 0.1

# ── DATASET ───────────────────────────────────────────────────
SAMPLE_PER_CLASS = 1200   # balansirano po tipu napada
INCLUDE_PORT     = False  # ⚠️ False = bez curenja informacije

print(f"Model:            {MODEL_ID}")
print(f"Effective batch:  {BATCH_SIZE * GRAD_ACCUM}")
print(f"Port u ulazu:     {INCLUDE_PORT}")


## 3. HuggingFace login

In [ ]:
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Prijavljen")
else:
    print("⚠️ Upiši HF_TOKEN u ćeliju 2!")


## 4. Upload više CICIDS2017 dana

**Odaberi najmanje 3–4 fajla** (Ctrl+klik za višestruki odabir):

| Fajl | Napadi |
|---|---|
| `Tuesday-WorkingHours` | FTP-Patator, SSH-Patator |
| `Wednesday-workingHours` | DoS Hulk, GoldenEye, slowloris, Slowhttptest |
| `Friday-...-PortScan` | PortScan |
| `Friday-...-DDos` | DDoS |
| `Thursday-...-WebAttacks` | SQL Injection, XSS, Brute Force |

Što više dana, to je zadatak realniji. Monday (samo BENIGN) nije nužan —
benign primjeri dolaze iz svakog fajla.


In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

dfs = []
for filename in uploaded:
    df = pd.read_csv(filename, low_memory=False)
    df = df.rename(columns={c: c.strip() for c in df.columns})
    labels = df["Label"].astype(str).str.strip().unique()
    print(f"✅ {filename}: {len(df):,} redova | {list(labels)}")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all["Label"] = df_all["Label"].astype(str).str.strip()

print(f"\n{'=' * 55}")
print(f"UKUPNO: {len(df_all):,} redova, {df_all['Label'].nunique()} različitih labela")
print(f"{'=' * 55}")
print(df_all["Label"].value_counts())


## 5. Priprema dataseta — bez curenja informacije

Tri izmjene u odnosu na v1.

**Port izbačen.** U v1 je odredišni port bio prvi podatak u ulaznom tekstu.
Kako FTP-Patator uvijek ide na 21, a SSH-Patator na 22, model je naučio
mapirati port na odgovor i ignorirati sve ostalo.

**Više značajki toka.** Umjesto 6, koristi se 14 numeričkih značajki koje
opisuju stvarno ponašanje toka — trajanje, omjer paketa, propusnost, IAT
statistike i TCP flagove.

**Predviđa se tip napada.** Izlazni JSON sad ima i `attack_type` uz
`risk_level`, pa se rezultat može izravno usporediti s XGBoost baselineom.


In [ ]:
import json, random
import numpy as np
from collections import Counter

# ── Značajke toka (BEZ porta) ─────────────────────────────────
FLOW_FEATURES = [
    "Flow Duration",
    "Total Fwd Packets", "Total Backward Packets",
    "Total Length of Fwd Packets", "Total Length of Bwd Packets",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "Flow Bytes/s", "Flow Packets/s",
    "Flow IAT Mean", "Flow IAT Std",
    "SYN Flag Count", "PSH Flag Count", "ACK Flag Count",
    "Down/Up Ratio", "Average Packet Size",
]

# ── Mapiranje labele → (tip napada, rizik, opis) ──────────────
ATTACK_MAP = {
    "BENIGN":                     ("BENIGN",       "LOW",    "Normalni mrežni promet bez znakova prijetnje."),
    "FTP-Patator":                ("FTP-Patator",  "HIGH",   "Brute force napad na FTP servis."),
    "SSH-Patator":                ("SSH-Patator",  "HIGH",   "Brute force napad na SSH servis."),
    "DoS Hulk":                   ("DoS",          "HIGH",   "DoS Hulk napad — preopterećenje HTTP zahtjevima."),
    "DoS GoldenEye":              ("DoS",          "HIGH",   "DoS GoldenEye napad — iscrpljivanje keep-alive veza."),
    "DoS slowloris":              ("DoS",          "HIGH",   "Slowloris napad — sporo držanje HTTP veza otvorenima."),
    "DoS Slowhttptest":           ("DoS",          "HIGH",   "Slow HTTP DoS napad."),
    "Heartbleed":                 ("DoS",          "HIGH",   "Heartbleed — pokušaj čitanja memorije servera."),
    "DDoS":                       ("DDoS",         "HIGH",   "Distribuirani DoS napad s više izvora."),
    "PortScan":                   ("PortScan",     "MEDIUM", "Skeniranje portova — izviđanje mreže."),
    "Bot":                        ("Botnet",       "HIGH",   "Botnet aktivnost — komunikacija s C&C serverom."),
    "Infiltration":               ("Infiltration", "HIGH",   "Infiltracija — neovlašteni pristup unutar mreže."),
    "Web Attack \u2013 Brute Force":  ("WebAttack", "HIGH",   "Brute force napad na web autentikaciju."),
    "Web Attack \u2013 XSS":          ("WebAttack", "HIGH",   "Cross-site scripting napad."),
    "Web Attack \u2013 Sql Injection":("WebAttack", "HIGH",   "SQL Injection napad na web aplikaciju."),
}

# CICIDS koristi i obican '-' umjesto en-dasha, pokrij oba
for k in list(ATTACK_MAP):
    if "\u2013" in k:
        ATTACK_MAP[k.replace("\u2013", "-")] = ATTACK_MAP[k]

INDICATORS = {
    "BENIGN":       ["Uobičajeno trajanje toka", "Uravnotežen omjer paketa"],
    "FTP-Patator":  ["Ponavljajuće kratke veze", "Visok broj pokušaja autentikacije"],
    "SSH-Patator":  ["Ponavljajuće kratke veze", "Uzastopni pokušaji prijave"],
    "DoS":          ["Visok broj zahtjeva po sekundi", "Kratki tokovi velike učestalosti"],
    "DDoS":         ["Vrlo visok SYN broj", "Kratko trajanje uz veliku propusnost"],
    "PortScan":     ["Minimalan prijenos podataka", "Vrlo kratki tokovi"],
    "Botnet":       ["Periodični obrazac prometa", "Pravilni vremenski intervali"],
    "WebAttack":    ["Neuobičajena veličina paketa", "Anomalan HTTP obrazac"],
    "Infiltration": ["Neuobičajen odlazni promet", "Netipično trajanje sesije"],
}

ACTIONS = {
    "BENIGN":       ["Nastaviti redovni monitoring"],
    "FTP-Patator":  ["Blokirati izvornu IP adresu", "Uvesti rate limiting na FTP"],
    "SSH-Patator":  ["Blokirati izvornu IP adresu", "Prijeći na autentikaciju ključem"],
    "DoS":          ["Aktivirati WAF", "Uvesti rate limiting"],
    "DDoS":         ["Aktivirati DDoS zaštitu", "Kontaktirati ISP"],
    "PortScan":     ["Blokirati izvornu IP adresu", "Pregledati firewall pravila"],
    "Botnet":       ["Izolirati zaraženi uređaj", "Blokirati C&C domenu"],
    "WebAttack":    ["Aktivirati WAF", "Pregledati validaciju ulaza"],
    "Infiltration": ["Izolirati pogođeni segment", "Pokrenuti forenzičku analizu"],
}


def build_instruction(row):
    label = str(row.get("Label", "BENIGN")).strip()
    attack, risk, summary = ATTACK_MAP.get(label, ("Unknown", "MEDIUM", f"Detektiran {label}."))

    parts = []
    if INCLUDE_PORT:                       # ⚠️ ostavljeno samo za usporedbu s v1
        parts.append(f"port={int(float(row.get('Destination Port', 0) or 0))}")

    for feat in FLOW_FEATURES:
        val = row.get(feat)
        if val is None:
            continue
        try:
            v = float(val)
        except (TypeError, ValueError):
            continue
        if np.isnan(v) or np.isinf(v):
            continue
        short = (feat.replace("Total ", "").replace("Length of ", "Len")
                     .replace("Packet", "Pkt").replace("Backward", "Bwd")
                     .replace("Forward", "Fwd").replace(" ", ""))
        parts.append(f"{short}={v:.1f}")

    proto = {"6": "TCP", "17": "UDP", "1": "ICMP"}.get(
        str(int(float(row.get("Protocol", 6) or 6))), "TCP")

    input_text = f"{proto} tok. " + ", ".join(parts)

    output = json.dumps({
        "attack_type":         attack,
        "risk_level":          risk,
        "summary":             summary,
        "key_indicators":      INDICATORS.get(attack, ["Anomalan obrazac toka"]),
        "recommended_actions": ACTIONS.get(attack, ["Istražiti aktivnost"]),
    }, ensure_ascii=False)

    text = (
        "### Instruction:\nAnaliziraj karakteristike ovog mrežnog toka, "
        "odredi tip napada i procijeni razinu rizika. Vrati ISKLJUČIVO JSON.\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n{output}"
    )
    return {"text": text, "attack_type": attack, "risk_level": risk, "raw_label": label}


# ── Balansirano uzorkovanje po TIPU napada ────────────────────
df_all["_attack"] = df_all["Label"].map(lambda l: ATTACK_MAP.get(str(l).strip(), ("Unknown",))[0])

records = []
for attack, group in df_all.groupby("_attack"):
    if attack == "Unknown":
        continue
    n = min(len(group), SAMPLE_PER_CLASS)
    for _, row in group.sample(n=n, random_state=42).iterrows():
        records.append(build_instruction(row))

random.seed(42)
random.shuffle(records)

print(f"✅ Dataset: {len(records):,} primjera\n")
print("Po tipu napada:")
for k, v in Counter(r["attack_type"] for r in records).most_common():
    print(f"  {k:<16} {v:>5}")
print("\nPo razini rizika:")
for k, v in Counter(r["risk_level"] for r in records).most_common():
    print(f"  {k:<16} {v:>5}")

print("\n" + "=" * 55)
print("PRIMJER ULAZA (provjeri da NEMA porta):")
print("=" * 55)
print(records[0]["text"][:600])


## 6. Train / Validation / Test split (80/10/10)

Stratificirano po tipu napada, pa svaki split ima sve klase u istom omjeru.


In [ ]:
from datasets import Dataset
from collections import defaultdict

by_class = defaultdict(list)
for r in records:
    by_class[r["attack_type"]].append(r)

train_recs, val_recs, test_recs = [], [], []
for attack, items in by_class.items():
    random.seed(42)
    random.shuffle(items)
    n_tr = int(len(items) * 0.80)
    n_va = int(len(items) * 0.10)
    train_recs += items[:n_tr]
    val_recs   += items[n_tr:n_tr + n_va]
    test_recs  += items[n_tr + n_va:]

for lst in (train_recs, val_recs, test_recs):
    random.shuffle(lst)

train_dataset = Dataset.from_list(train_recs)
val_dataset   = Dataset.from_list(val_recs)
test_dataset  = Dataset.from_list(test_recs)

print(f"Train:      {len(train_dataset):,}")
print(f"Validation: {len(val_dataset):,}")
print(f"Test:       {len(test_dataset):,}")
print("\nTest set po klasi:")
for k, v in Counter(test_dataset["attack_type"]).most_common():
    print(f"  {k:<16} {v:>4}")


## 7. Učitavanje modela (Unsloth, 4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch, time

t0 = time.perf_counter()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"✅ Učitan za {time.perf_counter()-t0:.1f}s | "
      f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


## 8. LoRA konfiguracija

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    target_modules = ["q_proj", "v_proj", "k_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)
model.print_trainable_parameters()


## 9. Fine-tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch, time

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    packing            = False,
    args = TrainingArguments(
        output_dir                  = f"./results/{MODEL_NAME}",
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        optim                       = "adamw_8bit",
        save_strategy               = "epoch",
        eval_strategy               = "epoch",
        logging_steps               = 50,
        learning_rate               = LEARNING_RATE,
        weight_decay                = 0.001,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        warmup_ratio                = WARMUP_RATIO,
        lr_scheduler_type           = "cosine",
        report_to                   = "none",
        load_best_model_at_end      = True,
    ),
)

print("🚀 Treniranje...")
t0 = time.perf_counter()
trainer.train()
TRAIN_MINUTES = round((time.perf_counter() - t0) / 60, 1)
print(f"\n✅ Gotovo za {TRAIN_MINUTES} min")


## 10. Spremanje na HuggingFace

**Pokreni odmah nakon treniranja.** Traje minutu i osigurava model
ako sesija kasnije padne — v1 je pao upravo u ovom koraku.


In [ ]:
SAVE_PATH = f"./{MODEL_NAME}"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Lokalno: {SAVE_PATH}")

if PUSH_TO_HUB and HF_USERNAME and HF_TOKEN:
    repo = f"{HF_USERNAME}/{MODEL_NAME}"
    model.push_to_hub(repo, token=HF_TOKEN)
    tokenizer.push_to_hub(repo, token=HF_TOKEN)
    print(f"✅ Hub: https://huggingface.co/{repo}")


## 11. Evaluacija — tip napada i razina rizika

Mjeri se oboje. `attack_type` je zahtjevniji zadatak i pravi pokazatelj
je li model nešto naučio; `risk_level` je grublja mjera koja se uspoređuje s v1.

Parsanje reže izlaz na prvom potpunom JSON objektu jer model nastavlja
generirati i nakon njega — bez toga `json.loads()` puca i accuracy ispadne 0%.


In [ ]:
import json, time, torch
from tqdm.auto import tqdm
from collections import defaultdict

FastLanguageModel.for_inference(model)

def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        p = text.split("```")
        if len(p) > 1:
            text = p[1][4:] if p[1].startswith("json") else p[1]
            text = text.strip()
    depth = 0; start = None
    for i, ch in enumerate(text):
        if ch == "{":
            if depth == 0: start = i
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0 and start is not None:
                return text[start:i+1]
    return None

EVAL_N = min(200, len(test_dataset))

atk_ok = risk_ok = parsed = 0
latencies = []
per_class = defaultdict(lambda: {"total": 0, "correct": 0})
confusion = defaultdict(int)

for item in tqdm(test_dataset.select(range(EVAL_N)), desc="Evaluiram"):
    prompt = item["text"].split("### Response:")[0] + "### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LENGTH).to("cuda")

    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=220, temperature=0.1,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    latencies.append((time.perf_counter() - t0) * 1000)

    gen = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                           skip_special_tokens=True)

    true_atk  = item["attack_type"]
    true_risk = item["risk_level"]
    per_class[true_atk]["total"] += 1

    raw = extract_json(gen)
    if raw is None:
        continue
    try:
        pred = json.loads(raw)
    except json.JSONDecodeError:
        continue

    parsed += 1
    p_atk  = pred.get("attack_type", "")
    p_risk = pred.get("risk_level", "")
    confusion[f"{true_atk}->{p_atk}"] += 1

    if p_atk == true_atk:
        atk_ok += 1
        per_class[true_atk]["correct"] += 1
    if p_risk == true_risk:
        risk_ok += 1

lat = sorted(latencies)
EVAL = {
    "attack_type_accuracy_pct": round(atk_ok  / EVAL_N * 100, 1),
    "risk_level_accuracy_pct":  round(risk_ok / EVAL_N * 100, 1),
    "json_parse_pct":           round(parsed  / EVAL_N * 100, 1),
    "n_samples":                EVAL_N,
    "latency_mean_ms":          round(sum(lat)/len(lat), 1),
    "latency_p95_ms":           round(lat[int(len(lat)*0.95)], 1),
    "per_class": {k: {**v, "accuracy_pct": round(v["correct"]/v["total"]*100, 1)}
                  for k, v in per_class.items()},
    "confusion": dict(sorted(confusion.items(), key=lambda x: -x[1])),
}

print(f"\n{'=' * 55}")
print(f"  Tip napada:    {EVAL['attack_type_accuracy_pct']}%")
print(f"  Razina rizika: {EVAL['risk_level_accuracy_pct']}%")
print(f"  JSON parse:    {EVAL['json_parse_pct']}%")
print(f"  Latencija:     {EVAL['latency_mean_ms']} ms (p95 {EVAL['latency_p95_ms']})")
print(f"{'=' * 55}\nPo klasi:")
for k, v in sorted(EVAL["per_class"].items(), key=lambda x: -x[1]["total"]):
    print(f"  {k:<16} {v['accuracy_pct']:>5}%  ({v['correct']}/{v['total']})")


## 12. Izvještaj za diplomski

Skuplja hiperparametre, loss po epohama, evaluaciju i podatke o datasetu
u jedan JSON. Preuzmi ga za svaki model — od tri takva fajla složit ćemo
usporedne tablice za evaluacijsko poglavlje.


In [ ]:
import json, torch, os
from datetime import datetime
from collections import Counter
from google.colab import files

hist = trainer.state.log_history
evals  = [e for e in hist if "eval_loss" in e]
trains = [e for e in hist if "loss" in e and "eval_loss" not in e]

epochs = []
for e in evals:
    ep = round(e.get("epoch", 0), 2)
    tl = [t for t in trains if abs(t.get("epoch", 0) - ep) < 0.5]
    epochs.append({
        "epoch":      ep,
        "train_loss": round(tl[-1]["loss"], 6) if tl else None,
        "eval_loss":  round(e["eval_loss"], 6),
    })

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

report = {
    "version":    "v2 — bez porta, više dana, predikcija tipa napada",
    "timestamp":  datetime.now().isoformat(),
    "model_id":   MODEL_ID,
    "model_name": MODEL_NAME,
    "gpu":        torch.cuda.get_device_name(0),
    "vram_peak_gb": round(torch.cuda.max_memory_allocated()/1024**3, 2),

    "hyperparameters": {
        "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "lora_dropout": LORA_DROPOUT,
        "max_seq_length": MAX_SEQ_LENGTH, "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM, "effective_batch": BATCH_SIZE * GRAD_ACCUM,
        "learning_rate": LEARNING_RATE, "num_epochs": NUM_EPOCHS,
        "warmup_ratio": WARMUP_RATIO,
    },
    "model_params": {
        "total": total, "trainable": trainable,
        "trainable_pct": round(trainable/total*100, 3),
    },
    "dataset": {
        "source": "CICIDS2017 (više dana)",
        "include_port": INCLUDE_PORT,
        "n_flow_features": len(FLOW_FEATURES),
        "sample_per_class": SAMPLE_PER_CLASS,
        "total_examples": len(records),
        "train": len(train_dataset), "validation": len(val_dataset),
        "test": len(test_dataset),
        "attack_distribution": dict(Counter(r["attack_type"] for r in records)),
        "risk_distribution":   dict(Counter(r["risk_level"]  for r in records)),
    },
    "training": {
        "total_steps": trainer.state.max_steps,
        "minutes": TRAIN_MINUTES,
        "epochs_log": epochs,
    },
    "evaluation": EVAL,
}

fname = f"report_{MODEL_NAME}.json"
with open(fname, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(json.dumps({k: v for k, v in report.items() if k != "evaluation"},
                 indent=2, ensure_ascii=False)[:1500])
files.download(fname)
print(f"\n✅ {fname} preuzet")


## 13. GGUF konverzija i download

In [ ]:
import glob, os
from google.colab import files

model.save_pretrained_gguf(f"{MODEL_NAME}-gguf", tokenizer,
                           quantization_method="q4_k_m")

found = [f for f in glob.glob(f"{MODEL_NAME}-gguf*/**/*.gguf", recursive=True)
         if "Q4_K_M" in f or "q4_k_m" in f]
if found:
    print(f"✅ {found[0]} ({os.path.getsize(found[0])/1024**3:.2f} GB)")
    files.download(found[0])
else:
    print("⚠️ GGUF nije pronađen")
    for f in glob.glob("**/*.gguf", recursive=True):
        print("  ", f)


## 14. Sljedeći model

Vrati se na **ćeliju 2**, promijeni `MODEL_ID` i `MODEL_NAME`, pa
Runtime → Restart session i pokreni ponovo. Restart je bitan da se
oslobodi VRAM od prethodnog modela.

| Model | MODEL_ID | Napomena |
|---|---|---|
| LLaMA 3.2 1B | `meta-llama/Llama-3.2-1B-Instruct` | najbrži |
| SmolLM2 1.7B | `HuggingFaceTB/SmolLM2-1.7B-Instruct` | — |
| Phi-3.5 Mini | `microsoft/Phi-3.5-mini-instruct` | `BATCH_SIZE=2`, `GRAD_ACCUM=8` |

Kad prikupiš sva tri `report_*.json` fajla, pošalji ih pa slažemo
usporedne tablice i grafove za evaluacijsko poglavlje.

**I ne zaboravi**: baseline modele (XGBoost, Random Forest) u projektu
treba ponovo istrenirati na istim danima — onih 99.4% s Tuesday fajla
ima isti problem kao v1, pa usporedba inače nije poštena.
